In [4]:
import json
import re
from pathlib import Path
import pandas as pd
from openai import OpenAI

# =========================
# 1. 路径设置
# =========================
BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")

INPUT_FILE = BASE_DIR / "post_base.csv"

POST_PILOT_FILE = BASE_DIR / "post_pilot_150_v2.csv"
RAW_OUTPUT_FILE = BASE_DIR / "post_topic_framework_v2_raw.txt"
JSON_OUTPUT_FILE = BASE_DIR / "post_topic_framework_v2.json"
TABLE_OUTPUT_FILE = BASE_DIR / "post_topic_framework_v2_table.csv"

# =========================
# 2. API 设置
# =========================
api_key = "xai-kw0TsJVQRUK00G0kGIlgo9mMvFLYaV6Tf6N23ZVTbpDWVj6LvgQMFzEq0Jln1iNqP8Dz7n98OXgRig61"

client = OpenAI(
    api_key=api_key,
    base_url="https://api.x.ai/v1"
)

MODEL_NAME = "grok-4-1-fast-non-reasoning"

# =========================
# 3. 读取 post_base
# =========================
df = pd.read_csv(INPUT_FILE, low_memory=False)
print("post_base 大小：", df.shape)

needed_cols = [
    "post_id",
    "post_text",
    "comment_count",
    "avg_sentiment_score",
    "blame_ratio",
    "main_blame_subject",
    "main_blame_object",
    "main_blame_intensity",
    "pos_ratio",
    "neg_ratio",
    "neu_ratio",
    "main_fine_grained_emotion"
]

for col in needed_cols:
    if col not in df.columns:
        df[col] = ""

df["post_text"] = df["post_text"].astype(str).str.strip()
df = df[df["post_text"] != ""].copy()

# 数值字段转换
num_cols = [
    "comment_count",
    "avg_sentiment_score",
    "blame_ratio",
    "pos_ratio",
    "neg_ratio",
    "neu_ratio"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("可用 post 数：", df.shape)

# =========================
# 4. 抽 150 条 pilot
#    尽量覆盖高评论、高负面、高 blame
# =========================
TARGET_N = 150
RANDOM_STATE = 42

sample_parts = []

# 1) 高评论帖子
high_comment = df.sort_values("comment_count", ascending=False).head(300)
sample_parts.append(high_comment.sample(n=min(50, len(high_comment)), random_state=RANDOM_STATE))

# 2) 高负面帖子
high_neg = df.sort_values("neg_ratio", ascending=False).head(300)
sample_parts.append(high_neg.sample(n=min(50, len(high_neg)), random_state=RANDOM_STATE + 1))

# 3) 高 blame 帖子
high_blame = df.sort_values("blame_ratio", ascending=False).head(300)
sample_parts.append(high_blame.sample(n=min(50, len(high_blame)), random_state=RANDOM_STATE + 2))

pilot_df = pd.concat(sample_parts, axis=0).drop_duplicates(subset=["post_id"])

# 不足 150 再随机补齐
if len(pilot_df) < TARGET_N:
    remain = df[~df["post_id"].isin(pilot_df["post_id"])]
    need = min(TARGET_N - len(pilot_df), len(remain))
    if need > 0:
        pilot_df = pd.concat([
            pilot_df,
            remain.sample(n=need, random_state=RANDOM_STATE + 3)
        ])

# 超过 150 裁剪
if len(pilot_df) > TARGET_N:
    pilot_df = pilot_df.sample(n=TARGET_N, random_state=RANDOM_STATE)

pilot_df = pilot_df.reset_index(drop=True)
pilot_df.to_csv(POST_PILOT_FILE, index=False, encoding="utf-8-sig")

print("post pilot 已保存：", POST_PILOT_FILE)
print("pilot 大小：", pilot_df.shape)

# =========================
# 5. 工具函数
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

def safe_number(x):
    try:
        return float(x)
    except:
        return None

def truncate_text(text, max_len=500):
    text = clean_text(text)
    return text if len(text) <= max_len else text[:max_len] + "..."

def extract_json_block(text):
    text = text.strip()
    match = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end+1]

    return text

# =========================
# 6. 组织 posts_data
# =========================
records = []

for _, row in pilot_df.iterrows():
    records.append({
        "post_id": clean_text(row.get("post_id", "")),
        "post_text": truncate_text(row.get("post_text", ""), 500),

        "comment_count": safe_number(row.get("comment_count", None)),
        "avg_sentiment_score": safe_number(row.get("avg_sentiment_score", None)),
        "blame_ratio": safe_number(row.get("blame_ratio", None)),

        "main_blame_subject": clean_text(row.get("main_blame_subject", "")),
        "main_blame_object": clean_text(row.get("main_blame_object", "")),
        "main_blame_intensity": clean_text(row.get("main_blame_intensity", "")),

        "pos_ratio": safe_number(row.get("pos_ratio", None)),
        "neg_ratio": safe_number(row.get("neg_ratio", None)),
        "neu_ratio": safe_number(row.get("neu_ratio", None)),
        "main_fine_grained_emotion": clean_text(row.get("main_fine_grained_emotion", "")),
    })

posts_data = json.dumps(records, ensure_ascii=False, indent=2)

# =========================
# 7. Prompt
# =========================
system_prompt = """你是一个专业的媒体数据分析师，擅长对反诈相关社群中的帖子进行话题聚类框架设计。

请基于 post_text 的语义内容为主，并结合该帖子下评论的聚合反馈特征：
- comment_count
- avg_sentiment_score
- blame_ratio
- main_blame_subject
- main_blame_object
- main_blame_intensity
- pos_ratio
- neg_ratio
- neu_ratio
- main_fine_grained_emotion

归纳出一套适用于 post-level 分析的话题框架。

注意：
1. 核心任务是识别“帖子在讲什么主题/事件/诈骗类型”。
2. post_text 是主要依据，评论情绪和 blame 特征用于辅助判断帖子引发的公众反应。
3. 不要只按情绪分类，例如不要只生成“负面帖子”“中性帖子”这种宽泛类别。
4. cluster 应该能解释帖子内容，也能帮助后续分析不同帖子话题引发的评论反应差异。
5. 请归纳 10-15 个有意义的 post topic cluster。
6. 输出必须是严格 JSON，不要添加任何解释文字。"""

user_prompt = f"""以下是 {len(records)} 条反诈相关帖子数据，请你进行 post topic framework 归纳。

请输出一个 JSON 对象，包含两个部分：

1. "clusters":
每个 cluster 包含：
- cluster_id: 整数，从 1 开始编号
- cluster_label: 简洁且有意义的话题名称（建议中文）
- cluster_keywords: 3-6 个核心关键词，数组格式
- short_description: 对该话题的简短描述
- inclusion_criteria: 哪类帖子应归入该 cluster
- exclusion_criteria: 哪类帖子不应归入该 cluster
- representative_post_ids: 2-5 个最能代表该话题的 post_id

2. "summary":
包含：
- suggested_cluster_count
- overlaps_or_confusions
- need_other_category
- overall_observation

帖子数据如下：
{posts_data}
"""

# =========================
# 8. 调用模型
# =========================
print("开始请求模型归纳 post topic framework v2...")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    max_completion_tokens=5000
)

raw_text = response.choices[0].message.content

with open(RAW_OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(raw_text)

print("原始返回已保存：", RAW_OUTPUT_FILE)

# =========================
# 9. 解析 JSON
# =========================
json_text = extract_json_block(raw_text)
parsed = json.loads(json_text)

with open(JSON_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(parsed, f, ensure_ascii=False, indent=2)

print("JSON 已保存：", JSON_OUTPUT_FILE)

# =========================
# 10. 转成表格
# =========================
clusters = parsed.get("clusters", [])
summary = parsed.get("summary", {})

cluster_rows = []

for c in clusters:
    cluster_rows.append({
        "cluster_id": c.get("cluster_id"),
        "cluster_label": c.get("cluster_label"),
        "cluster_keywords": ", ".join(c.get("cluster_keywords", [])) if isinstance(c.get("cluster_keywords"), list) else c.get("cluster_keywords"),
        "short_description": c.get("short_description"),
        "inclusion_criteria": c.get("inclusion_criteria"),
        "exclusion_criteria": c.get("exclusion_criteria"),
        "representative_post_ids": ", ".join(c.get("representative_post_ids", [])) if isinstance(c.get("representative_post_ids"), list) else c.get("representative_post_ids"),
    })

cluster_df = pd.DataFrame(cluster_rows)
cluster_df.to_csv(TABLE_OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("post topic framework v2 表已保存：", TABLE_OUTPUT_FILE)

print("\n=== Post Topic Framework v2 简要结果 ===")
print("建议 cluster 数：", summary.get("suggested_cluster_count"))
print("是否需要 other 类：", summary.get("need_other_category"))
print("总体观察：", summary.get("overall_observation"))

print("\n候选 post clusters：")
for c in clusters:
    print(f"- [{c.get('cluster_id')}] {c.get('cluster_label')} | 关键词: {c.get('cluster_keywords')}")

post_base 大小： (8307, 17)
可用 post 数： (8307, 17)
post pilot 已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_pilot_150_v2.csv
pilot 大小： (150, 17)
开始请求模型归纳 post topic framework v2...
原始返回已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_topic_framework_v2_raw.txt
JSON 已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_topic_framework_v2.json
post topic framework v2 表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_topic_framework_v2_table.csv

=== Post Topic Framework v2 简要结果 ===
建议 cluster 数： 15
是否需要 other 类： 否，15个cluster覆盖所有帖子；少数政治讽刺（如PTT_1764654917）可归入社会现象讨论
总体观察： PTT新闻帖评论高neg/anger/blame诈骗集团，常引发司法讨论；Reddit多个人经历求证，neu_ratio较高；Facebook简短曝光求证居多，blame强但comment_count低。诈骗类型多样，电话/平台/情感最常见，公众反应以愤怒为主导。

候选 post clusters：
- [1] 司法宽纵车手 | 关键词: ['车手', '交保', '法院', '放虎归山', '再犯']
- [2] 个人求证诈骗 | 关键词: ['这是诈骗吗', '求问', '可疑', '收到消息', '请问']
- [3] 浪漫杀猪盘 | 关键词: ['恋爱诈骗', '杀猪盘', '网络恋情', '假男友', '老人中招']
- [4] 假冒银行电话 

In [5]:
import json
import re
from pathlib import Path
import pandas as pd
from openai import OpenAI

# =========================
# 1. 路径设置
# =========================
BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")

INPUT_FILE = BASE_DIR / "post_base.csv"

POST_PILOT_FILE = BASE_DIR / "post_reaction_pilot_150.csv"
RAW_OUTPUT_FILE = BASE_DIR / "post_reaction_framework_raw.txt"
JSON_OUTPUT_FILE = BASE_DIR / "post_reaction_framework.json"
TABLE_OUTPUT_FILE = BASE_DIR / "post_reaction_framework_table.csv"

# =========================
# 2. API 设置
# =========================
api_key = "xai-kw0TsJVQRUK00G0kGIlgo9mMvFLYaV6Tf6N23ZVTbpDWVj6LvgQMFzEq0Jln1iNqP8Dz7n98OXgRig61"

client = OpenAI(
    api_key=api_key,
    base_url="https://api.x.ai/v1"
)

MODEL_NAME = "grok-4-1-fast-non-reasoning"

# =========================
# 3. 读取 post_base
# =========================
df = pd.read_csv(INPUT_FILE, low_memory=False)
print("post_base 大小：", df.shape)

needed_cols = [
    "post_id",
    "post_text",
    "comment_count",
    "avg_sentiment_score",
    "blame_ratio",
    "main_blame_subject",
    "main_blame_object",
    "main_blame_intensity",
    "pos_ratio",
    "neg_ratio",
    "neu_ratio",
    "main_fine_grained_emotion"
]

for col in needed_cols:
    if col not in df.columns:
        df[col] = ""

df["post_text"] = df["post_text"].astype(str).str.strip()
df = df[df["post_text"] != ""].copy()

num_cols = [
    "comment_count",
    "avg_sentiment_score",
    "blame_ratio",
    "pos_ratio",
    "neg_ratio",
    "neu_ratio"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("可用 post 数：", df.shape)

# =========================
# 4. 抽 150 条 pilot
# 覆盖：高评论 / 高负面 / 高指责
# =========================
TARGET_N = 150
RANDOM_STATE = 42

sample_parts = []

high_comment = df.sort_values("comment_count", ascending=False).head(300)
sample_parts.append(high_comment.sample(n=min(50, len(high_comment)), random_state=RANDOM_STATE))

high_neg = df.sort_values("neg_ratio", ascending=False).head(300)
sample_parts.append(high_neg.sample(n=min(50, len(high_neg)), random_state=RANDOM_STATE + 1))

high_blame = df.sort_values("blame_ratio", ascending=False).head(300)
sample_parts.append(high_blame.sample(n=min(50, len(high_blame)), random_state=RANDOM_STATE + 2))

pilot_df = pd.concat(sample_parts, axis=0).drop_duplicates(subset=["post_id"])

if len(pilot_df) < TARGET_N:
    remain = df[~df["post_id"].isin(pilot_df["post_id"])]
    need = min(TARGET_N - len(pilot_df), len(remain))
    if need > 0:
        pilot_df = pd.concat([
            pilot_df,
            remain.sample(n=need, random_state=RANDOM_STATE + 3)
        ])

if len(pilot_df) > TARGET_N:
    pilot_df = pilot_df.sample(n=TARGET_N, random_state=RANDOM_STATE)

pilot_df = pilot_df.reset_index(drop=True)
pilot_df.to_csv(POST_PILOT_FILE, index=False, encoding="utf-8-sig")

print("post reaction pilot 已保存：", POST_PILOT_FILE)
print("pilot 大小：", pilot_df.shape)

# =========================
# 5. 工具函数
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

def safe_number(x):
    try:
        return float(x)
    except:
        return None

def truncate_text(text, max_len=450):
    text = clean_text(text)
    return text if len(text) <= max_len else text[:max_len] + "..."

def extract_json_block(text):
    text = text.strip()
    match = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end+1]

    return text

# =========================
# 6. 组织 posts_data
# =========================
records = []

for _, row in pilot_df.iterrows():
    records.append({
        "post_id": clean_text(row.get("post_id", "")),
        "post_text": truncate_text(row.get("post_text", ""), 450),
        "comment_count": safe_number(row.get("comment_count", None)),
        "avg_sentiment_score": safe_number(row.get("avg_sentiment_score", None)),
        "blame_ratio": safe_number(row.get("blame_ratio", None)),
        "main_blame_subject": clean_text(row.get("main_blame_subject", "")),
        "main_blame_object": clean_text(row.get("main_blame_object", "")),
        "main_blame_intensity": clean_text(row.get("main_blame_intensity", "")),
        "pos_ratio": safe_number(row.get("pos_ratio", None)),
        "neg_ratio": safe_number(row.get("neg_ratio", None)),
        "neu_ratio": safe_number(row.get("neu_ratio", None)),
        "main_fine_grained_emotion": clean_text(row.get("main_fine_grained_emotion", "")),
    })

posts_data = json.dumps(records, ensure_ascii=False, indent=2)

# =========================
# 7. Prompt：情绪 + blame 反应型 cluster
# =========================
system_prompt = """你是一个专业的媒体数据分析师，擅长基于评论情绪和责任归因反馈，对反诈相关帖子进行 reaction-based clustering。

请综合考虑：
- post_text
- comment_count
- avg_sentiment_score
- blame_ratio
- main_blame_subject
- main_blame_object
- main_blame_intensity
- pos_ratio
- neg_ratio
- neu_ratio
- main_fine_grained_emotion

你的任务不是传统帖子主题聚类，而是识别“该帖子引发了什么类型的公众反应”。

请尝试将帖子归纳为 10-15 个反应型 cluster，例如但不限于：
1. 指责平台/监管失职
2. 指责受害者本人轻信/贪心
3. 指责骗子/作恶方
4. 一般性愤怒谴责
5. 表达同情/安慰/支持
6. 防骗建议与风险提醒
7. 分享相似经历/亲身案例
8. 质疑真实性/怀疑内容
9. 调侃/嘲讽/mock
10. 中性信息补充/事实说明
11. 呼吁报警/维权/求助
12. 对社会环境/制度机制的批评
13. 提问/求证/寻求解释
14. 情绪化感叹但无明确责任指向
15. 其他/混合类

注意：
1. 聚类重点是公众反应模式、情绪倾向、责任归因方式和表达立场。
2. post_text 用于理解帖子背景，评论聚合特征用于判断该帖引发的反应。
3. 不要只输出“负面帖子”“中性帖子”这种过泛类别。
4. cluster 之间要边界清晰，避免重复。
5. 输出必须是严格 JSON，不要添加任何解释文字。"""

user_prompt = f"""以下是 {len(records)} 条反诈相关帖子数据。请基于情绪、指责、评论反馈和帖子文本背景，归纳 reaction-based post cluster framework。

请输出一个 JSON 对象，包含两个部分：

1. "clusters":
每个 cluster 包含：
- cluster_id: 整数，从 1 开始编号
- cluster_label: 简洁且有意义的反应型 cluster 名称（建议中文）
- cluster_keywords: 3-6 个核心关键词，数组格式
- short_description: 该 cluster 的简短描述
- inclusion_criteria: 哪类帖子应归入该 cluster
- exclusion_criteria: 哪类帖子不应归入该 cluster
- representative_post_ids: 2-5 个最能代表该 cluster 的 post_id

2. "summary":
包含：
- suggested_cluster_count
- overlaps_or_confusions
- need_other_category
- overall_observation

帖子数据如下：
{posts_data}
"""

# =========================
# 8. 调用模型
# =========================
print("开始请求模型归纳 post reaction framework...")

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    max_completion_tokens=5000
)

raw_text = response.choices[0].message.content

with open(RAW_OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(raw_text)

print("原始返回已保存：", RAW_OUTPUT_FILE)

# =========================
# 9. 解析 JSON
# =========================
json_text = extract_json_block(raw_text)
parsed = json.loads(json_text)

with open(JSON_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(parsed, f, ensure_ascii=False, indent=2)

print("JSON 已保存：", JSON_OUTPUT_FILE)

# =========================
# 10. 转成表格
# =========================
clusters = parsed.get("clusters", [])
summary = parsed.get("summary", {})

cluster_rows = []

for c in clusters:
    cluster_rows.append({
        "cluster_id": c.get("cluster_id"),
        "cluster_label": c.get("cluster_label"),
        "cluster_keywords": ", ".join(c.get("cluster_keywords", [])) if isinstance(c.get("cluster_keywords"), list) else c.get("cluster_keywords"),
        "short_description": c.get("short_description"),
        "inclusion_criteria": c.get("inclusion_criteria"),
        "exclusion_criteria": c.get("exclusion_criteria"),
        "representative_post_ids": ", ".join(c.get("representative_post_ids", [])) if isinstance(c.get("representative_post_ids"), list) else c.get("representative_post_ids"),
    })

cluster_df = pd.DataFrame(cluster_rows)
cluster_df.to_csv(TABLE_OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("post reaction framework 表已保存：", TABLE_OUTPUT_FILE)

print("\n=== Post Reaction Framework 简要结果 ===")
print("建议 cluster 数：", summary.get("suggested_cluster_count"))
print("是否需要 other 类：", summary.get("need_other_category"))
print("总体观察：", summary.get("overall_observation"))

print("\n候选 post reaction clusters：")
for c in clusters:
    print(f"- [{c.get('cluster_id')}] {c.get('cluster_label')} | 关键词: {c.get('cluster_keywords')}")

post_base 大小： (8307, 17)
可用 post 数： (8307, 17)
post reaction pilot 已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_reaction_pilot_150.csv
pilot 大小： (150, 17)
开始请求模型归纳 post reaction framework...
原始返回已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_reaction_framework_raw.txt
JSON 已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_reaction_framework.json
post reaction framework 表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/post_reaction_framework_table.csv

=== Post Reaction Framework 简要结果 ===
建议 cluster 数： 12
是否需要 other 类： False
总体观察： 数据以负面情绪为主(neg_ratio平均>0.7)，PTT帖子comment多、高强度愤怒指向司法/骗子；Reddit多求证/个人故事，中性较高；Facebook简短检举/提醒为主。诈骗类型多样(投资/爱情/包裹)，反应模式高度一致于谴责与警示。

候选 post reaction clusters：
- [1] 指责骗子/诈骗集团 | 关键词: ['诈骗集团', '骗徒', '车手', '杀猪盘', '假账号']
- [2] 指责司法/法院轻纵 | 关键词: ['法院', '交保', '放虎归山', '羁押', '鞭刑']
- [3] 质疑真实性/求证诈骗 | 关键词: ['这是诈骗吗', 'am I getting scammed', '求证', '是真的吗']
- [4] 分享个人受害经历 | 关键词: ['

In [7]:
import json
import re
import math
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

# =====================================================
# 1. 路径设置
# =====================================================
BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")

POST_BASE_FILE = BASE_DIR / "post_base.csv"
FRAMEWORK_FILE = BASE_DIR / "post_reaction_framework_table.csv"

RESULT_FILE = BASE_DIR / "trial_1000_post_reaction_results.csv"
COUNT_FILE = BASE_DIR / "trial_1000_post_reaction_count_summary.csv"
SENTIMENT_FILE = BASE_DIR / "trial_1000_post_reaction_sentiment_summary.csv"
BLAME_FILE = BASE_DIR / "trial_1000_post_reaction_blame_summary.csv"

RAW_DIR = BASE_DIR / "trial_1000_post_reaction_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# 2. API 设置
# =====================================================
api_key = "xai-kw0TsJVQRUK00G0kGIlgo9mMvFLYaV6Tf6N23ZVTbpDWVj6LvgQMFzEq0Jln1iNqP8Dz7n98OXgRig61"

client = OpenAI(
    api_key=api_key,
    base_url="https://api.x.ai/v1"
)

MODEL_NAME = "grok-4-1-fast-non-reasoning"

# =====================================================
# 3. 读取数据
# =====================================================
post_df = pd.read_csv(POST_BASE_FILE, low_memory=False)
framework_df = pd.read_csv(FRAMEWORK_FILE, low_memory=False)

print("post_base:", post_df.shape)
print("framework:", framework_df.shape)

# 先随机抽 1000 条测试
TRIAL_N = 1000
if len(post_df) > TRIAL_N:
    post_df = post_df.sample(n=TRIAL_N, random_state=42).copy()
else:
    post_df = post_df.copy()

post_df = post_df.reset_index(drop=True)
print("试跑 post 数:", post_df.shape)

# =====================================================
# 4. 工具函数
# =====================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

def safe_number(x):
    if pd.isna(x):
        return None
    try:
        return float(x)
    except Exception:
        return None

def truncate_text(text, max_len=450):
    text = clean_text(text)
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."

def extract_json_block(text):
    text = text.strip()

    match = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    match = re.search(r"```\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]

    return text

def mode_nonempty(series):
    s = series.dropna().astype(str).str.strip()
    s = s[~s.isin(["", "nan", "None", "null"])]
    if len(s) == 0:
        return ""
    return s.value_counts().index[0]

# =====================================================
# 5. 整理 framework
# =====================================================
framework_records = []

for _, row in framework_df.iterrows():
    framework_records.append({
        "cluster_id": int(row["cluster_id"]),
        "cluster_label": clean_text(row["cluster_label"]),
        "cluster_keywords": clean_text(row["cluster_keywords"]),
        "short_description": clean_text(row["short_description"]),
        "inclusion_criteria": clean_text(row["inclusion_criteria"]),
        "exclusion_criteria": clean_text(row["exclusion_criteria"]),
    })

cluster_map = {
    int(row["cluster_id"]): clean_text(row["cluster_label"])
    for _, row in framework_df.iterrows()
}

keyword_map = {
    int(row["cluster_id"]): clean_text(row["cluster_keywords"])
    for _, row in framework_df.iterrows()
}

framework_text = json.dumps(framework_records, ensure_ascii=False, indent=2)

# =====================================================
# 6. Prompt
# =====================================================
system_prompt = """你是一个专业的媒体数据分析师，擅长根据既定的 post reaction cluster framework，对反诈相关帖子进行分类。

请严格按照给定的 framework 对每条帖子进行单标签分类。

注意：
1. post_text 是主要依据。
2. comment_count、avg_sentiment_score、blame_ratio、pos_ratio、neg_ratio、neu_ratio、main_blame_object、main_fine_grained_emotion 是辅助依据。
3. 这次分类重点是帖子引发的公众反应模式、情绪倾向和责任归因方式。
4. 每条 post 只能归入一个最合适的 cluster。
5. 不要新建 cluster，不要修改 cluster_id。
6. 输出必须是严格 JSON，不要添加解释文字。"""

def make_user_prompt(records):
    return f"""以下是固定好的 post reaction cluster framework：

{framework_text}

请根据该 framework，对下面 {len(records)} 条帖子逐条分类。

每条帖子输出：
- post_id
- cluster_id
- confidence: 0-1 之间的小数

帖子数据：
{json.dumps(records, ensure_ascii=False, indent=2)}

请输出严格 JSON，格式如下：
{{
  "results": [
    {{
      "post_id": "...",
      "cluster_id": 1,
      "confidence": 0.92
    }}
  ]
}}
"""

# =====================================================
# 7. 批量分类
# =====================================================
BATCH_SIZE = 20
MAX_RETRY = 3
all_results = []

num_batches = math.ceil(len(post_df) / BATCH_SIZE)
print("总批次数:", num_batches)

for batch_idx in range(num_batches):
    start = batch_idx * BATCH_SIZE
    end = min((batch_idx + 1) * BATCH_SIZE, len(post_df))
    batch_df = post_df.iloc[start:end].copy()

    records = []
    for _, row in batch_df.iterrows():
        records.append({
            "post_id": clean_text(row.get("post_id", "")),
            "post_text": truncate_text(row.get("post_text", ""), 450),
            "comment_count": safe_number(row.get("comment_count", None)),
            "avg_sentiment_score": safe_number(row.get("avg_sentiment_score", None)),
            "blame_ratio": safe_number(row.get("blame_ratio", None)),
            "main_blame_subject": clean_text(row.get("main_blame_subject", "")),
            "main_blame_object": clean_text(row.get("main_blame_object", "")),
            "main_blame_intensity": clean_text(row.get("main_blame_intensity", "")),
            "pos_ratio": safe_number(row.get("pos_ratio", None)),
            "neg_ratio": safe_number(row.get("neg_ratio", None)),
            "neu_ratio": safe_number(row.get("neu_ratio", None)),
            "main_fine_grained_emotion": clean_text(row.get("main_fine_grained_emotion", "")),
        })

    user_prompt = make_user_prompt(records)

    print(f"开始 batch {batch_idx + 1}/{num_batches}")

    success = False
    for attempt in range(MAX_RETRY):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_completion_tokens=3000
            )

            raw_text = response.choices[0].message.content

            raw_file = RAW_DIR / f"batch_{batch_idx + 1:03d}_raw.txt"
            with open(raw_file, "w", encoding="utf-8") as f:
                f.write(raw_text)

            parsed = json.loads(extract_json_block(raw_text))
            results = parsed.get("results", [])

            for item in results:
                cid = int(item["cluster_id"])
                all_results.append({
                    "post_id": clean_text(item["post_id"]),
                    "post_cluster_id": cid,
                    "post_cluster_label": cluster_map.get(cid, ""),
                    "post_cluster_keywords": keyword_map.get(cid, ""),
                    "confidence": item.get("confidence", None)
                })

            print(f"batch {batch_idx + 1} 完成，返回 {len(results)} 条")
            success = True
            break

        except Exception as e:
            print(f"batch {batch_idx + 1} 第 {attempt + 1} 次失败：{e}")
            time.sleep(2)

    if not success:
        raise RuntimeError(f"batch {batch_idx + 1} 连续失败")

# =====================================================
# 8. 合并结果
# =====================================================
result_df = pd.DataFrame(all_results)
result_df = result_df.drop_duplicates(subset=["post_id"])

final_df = post_df.merge(result_df, on="post_id", how="left")

final_df.to_csv(RESULT_FILE, index=False, encoding="utf-8-sig")
print("post 试跑分类结果已保存:", RESULT_FILE)

# =====================================================
# 9. 数量汇总
# =====================================================
count_df = (
    final_df.groupby(["post_cluster_id", "post_cluster_label"], dropna=False)
    .agg(
        post_count=("post_id", "count"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
    .sort_values("post_count", ascending=False)
)

count_df["ratio"] = count_df["post_count"] / len(final_df)
count_df.to_csv(COUNT_FILE, index=False, encoding="utf-8-sig")
print("post cluster 数量汇总已保存:", COUNT_FILE)

# =====================================================
# 10. 情感汇总
# =====================================================
sentiment_summary = (
    final_df.groupby(["post_cluster_id", "post_cluster_label"], dropna=False)
    .agg(
        post_count=("post_id", "count"),
        avg_sentiment_score=("avg_sentiment_score", "mean"),
        avg_pos_ratio=("pos_ratio", "mean"),
        avg_neg_ratio=("neg_ratio", "mean"),
        avg_neu_ratio=("neu_ratio", "mean"),
        main_fine_grained_emotion=("main_fine_grained_emotion", mode_nonempty)
    )
    .reset_index()
    .sort_values("post_cluster_id")
)

sentiment_summary.to_csv(SENTIMENT_FILE, index=False, encoding="utf-8-sig")
print("post cluster 情感汇总已保存:", SENTIMENT_FILE)

# =====================================================
# 11. blame 汇总
# =====================================================
blame_summary = (
    final_df.groupby(["post_cluster_id", "post_cluster_label"], dropna=False)
    .agg(
        post_count=("post_id", "count"),
        avg_blame_ratio=("blame_ratio", "mean"),
        main_blame_subject=("main_blame_subject", mode_nonempty),
        main_blame_object=("main_blame_object", mode_nonempty),
        main_blame_intensity=("main_blame_intensity", mode_nonempty)
    )
    .reset_index()
    .sort_values("post_cluster_id")
)

blame_summary.to_csv(BLAME_FILE, index=False, encoding="utf-8-sig")
print("post cluster blame 汇总已保存:", BLAME_FILE)

# =====================================================
# 12. 打印结果
# =====================================================
print("\n=== post reaction cluster 1000 条试跑完成 ===")
print("总 post 数:", len(final_df))
print("已分类 post 数:", final_df["post_cluster_id"].notna().sum())
print("未分类 post 数:", final_df["post_cluster_id"].isna().sum())

print("\ncluster 分布:")
print(count_df)

post_base: (8307, 17)
framework: (12, 7)
试跑 post 数: (1000, 17)
总批次数: 50
开始 batch 1/50
batch 1 完成，返回 20 条
开始 batch 2/50
batch 2 完成，返回 20 条
开始 batch 3/50
batch 3 完成，返回 20 条
开始 batch 4/50
batch 4 完成，返回 20 条
开始 batch 5/50
batch 5 完成，返回 20 条
开始 batch 6/50
batch 6 完成，返回 20 条
开始 batch 7/50
batch 7 完成，返回 20 条
开始 batch 8/50
batch 8 完成，返回 20 条
开始 batch 9/50
batch 9 完成，返回 20 条
开始 batch 10/50
batch 10 完成，返回 20 条
开始 batch 11/50
batch 11 完成，返回 20 条
开始 batch 12/50
batch 12 完成，返回 20 条
开始 batch 13/50
batch 13 完成，返回 20 条
开始 batch 14/50
batch 14 完成，返回 20 条
开始 batch 15/50
batch 15 完成，返回 20 条
开始 batch 16/50
batch 16 完成，返回 20 条
开始 batch 17/50
batch 17 完成，返回 20 条
开始 batch 18/50
batch 18 完成，返回 20 条
开始 batch 19/50
batch 19 完成，返回 20 条
开始 batch 20/50
batch 20 完成，返回 20 条
开始 batch 21/50
batch 21 完成，返回 20 条
开始 batch 22/50
batch 22 完成，返回 20 条
开始 batch 23/50
batch 23 完成，返回 20 条
开始 batch 24/50
batch 24 完成，返回 20 条
开始 batch 25/50
batch 25 完成，返回 20 条
开始 batch 26/50
batch 26 完成，返回 20 条
开始 batch 27/50
batch 27 完成，返回 20 条
开

In [8]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")
RESULT_FILE = BASE_DIR / "trial_1000_post_reaction_results.csv"

df = pd.read_csv(RESULT_FILE, low_memory=False)

missing = df[df["post_cluster_id"].isna()].copy()

print("没分到的 post 数：", len(missing))
print(missing[["post_id", "post_text", "comment_count", "avg_sentiment_score", "blame_ratio"]].head(20))

missing.to_csv(BASE_DIR / "trial_1000_missing_post.csv", index=False, encoding="utf-8-sig")
print("已保存：trial_1000_missing_post.csv")

没分到的 post 数： 0
Empty DataFrame
Columns: [post_id, post_text, comment_count, avg_sentiment_score, blame_ratio]
Index: []
已保存：trial_1000_missing_post.csv


In [19]:
import json
import re
import math
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

# =====================================================
# 1. 路径设置
# =====================================================
BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")

POST_BASE_FILE = BASE_DIR / "post_base.csv"
FRAMEWORK_FILE = BASE_DIR / "post_reaction_framework_table.csv"

BATCH_RESULTS_DIR = BASE_DIR / "full_post_reaction_batch_results"
RAW_DIR = BASE_DIR / "full_post_reaction_raw"
RECOVERY_DIR = BASE_DIR / "full_post_reaction_recovery"

BATCH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

FINAL_RESULT_FILE = BASE_DIR / "full_post_reaction_results.csv"
COUNT_FILE = BASE_DIR / "full_post_reaction_count_summary.csv"
SENTIMENT_FILE = BASE_DIR / "full_post_reaction_sentiment_summary.csv"
BLAME_FILE = BASE_DIR / "full_post_reaction_blame_summary.csv"
MISSING_FILE = BASE_DIR / "full_post_reaction_missing_posts.csv"

# =====================================================
# 2. API 设置
# =====================================================
api_key = "xai-kw0TsJVQRUK00G0kGIlgo9mMvFLYaV6Tf6N23ZVTbpDWVj6LvgQMFzEq0Jln1iNqP8Dz7n98OXgRig61"

client = OpenAI(
    api_key=api_key,
    base_url="https://api.x.ai/v1"
)

MODEL_NAME = "grok-4-1-fast-non-reasoning"

# =====================================================
# 3. 参数设置
# =====================================================
BATCH_SIZE = 8
RECOVERY_BATCH_SIZE = 8
MAX_RETRY = 3
SLEEP_SECONDS = 2
MAX_TEXT_LEN = 450

# =====================================================
# 4. 读取数据
# =====================================================
post_df = pd.read_csv(POST_BASE_FILE, low_memory=False)
framework_df = pd.read_csv(FRAMEWORK_FILE, low_memory=False)

print("post_base:", post_df.shape)
print("framework:", framework_df.shape)

needed_cols = [
    "post_id", "post_text",
    "comment_count", "avg_sentiment_score", "blame_ratio",
    "main_blame_subject", "main_blame_object", "main_blame_intensity",
    "pos_ratio", "neg_ratio", "neu_ratio", "main_fine_grained_emotion"
]

for col in needed_cols:
    if col not in post_df.columns:
        post_df[col] = ""

post_df["post_id"] = post_df["post_id"].astype(str).str.strip()
post_df["post_text"] = post_df["post_text"].astype(str).str.strip()

post_df = post_df[
    (post_df["post_id"] != "") &
    (post_df["post_text"] != "")
].drop_duplicates(subset=["post_id"]).reset_index(drop=True)

print("清洗后 post 数:", len(post_df))

# =====================================================
# 5. 工具函数
# =====================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

def safe_number(x):
    if pd.isna(x):
        return None
    try:
        return float(x)
    except Exception:
        return None

def truncate_text(text, max_len=450):
    text = clean_text(text)
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."

def extract_json_block(text):
    text = text.strip()

    match = re.search(r"```json\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    match = re.search(r"```\s*(\{.*\})\s*```", text, re.DOTALL)
    if match:
        return match.group(1)

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]

    return text

def mode_nonempty(series):
    s = series.dropna().astype(str).str.strip()
    s = s[~s.isin(["", "nan", "None", "null"])]
    if len(s) == 0:
        return ""
    return s.value_counts().index[0]

def build_post_records(batch_df):
    records = []
    for _, row in batch_df.iterrows():
        records.append({
            "post_id": clean_text(row.get("post_id", "")),
            "post_text": truncate_text(row.get("post_text", ""), MAX_TEXT_LEN),
            "comment_count": safe_number(row.get("comment_count", None)),
            "avg_sentiment_score": safe_number(row.get("avg_sentiment_score", None)),
            "blame_ratio": safe_number(row.get("blame_ratio", None)),
            "main_blame_subject": clean_text(row.get("main_blame_subject", "")),
            "main_blame_object": clean_text(row.get("main_blame_object", "")),
            "main_blame_intensity": clean_text(row.get("main_blame_intensity", "")),
            "pos_ratio": safe_number(row.get("pos_ratio", None)),
            "neg_ratio": safe_number(row.get("neg_ratio", None)),
            "neu_ratio": safe_number(row.get("neu_ratio", None)),
            "main_fine_grained_emotion": clean_text(row.get("main_fine_grained_emotion", "")),
        })
    return records

# =====================================================
# 6. 整理 framework
# =====================================================
framework_records = []
for _, row in framework_df.iterrows():
    framework_records.append({
        "cluster_id": int(row["cluster_id"]),
        "cluster_label": clean_text(row["cluster_label"]),
        "cluster_keywords": clean_text(row["cluster_keywords"]),
        "short_description": clean_text(row["short_description"]),
        "inclusion_criteria": clean_text(row["inclusion_criteria"]),
        "exclusion_criteria": clean_text(row["exclusion_criteria"]),
    })

cluster_map = {
    int(row["cluster_id"]): clean_text(row["cluster_label"])
    for _, row in framework_df.iterrows()
}

keyword_map = {
    int(row["cluster_id"]): clean_text(row["cluster_keywords"])
    for _, row in framework_df.iterrows()
}

framework_text = json.dumps(framework_records, ensure_ascii=False, indent=2)

system_prompt = """你是一个专业的媒体数据分析师，擅长根据既定的 post reaction cluster framework，对反诈相关帖子进行分类。

请严格按照给定的 framework 对每条帖子进行单标签分类。

注意：
1. post_text 是主要依据。
2. comment_count、avg_sentiment_score、blame_ratio、pos_ratio、neg_ratio、neu_ratio、main_blame_object、main_fine_grained_emotion 是辅助依据。
3. 这次分类重点是帖子引发的公众反应模式、情绪倾向和责任归因方式。
4. 每条 post 只能归入一个最合适的 cluster。
5. 不要新建 cluster，不要修改 cluster_id。
6. 输出必须是严格 JSON，不要添加解释文字。"""

def make_user_prompt(records):
    return f"""以下是固定好的 post reaction cluster framework：

{framework_text}

请根据该 framework，对下面 {len(records)} 条帖子逐条分类。

每条帖子输出：
- post_id
- cluster_id
- confidence: 0-1 之间的小数

帖子数据：
{json.dumps(records, ensure_ascii=False, indent=2)}

请输出严格 JSON，格式如下：
{{
  "results": [
    {{
      "post_id": "...",
      "cluster_id": 1,
      "confidence": 0.92
    }}
  ]
}}
"""

def parse_results(raw_text):
    parsed = json.loads(extract_json_block(raw_text))
    results = parsed.get("results", [])

    rows = []
    for item in results:
        try:
            cid = int(item.get("cluster_id"))
        except Exception:
            continue

        if cid not in cluster_map:
            continue

        rows.append({
            "post_id": clean_text(item.get("post_id", "")),
            "post_cluster_id": cid,
            "post_cluster_label": cluster_map.get(cid, ""),
            "post_cluster_keywords": keyword_map.get(cid, ""),
            "confidence": item.get("confidence", None)
        })
    return rows

# =====================================================
# 7. 全量批量分类，支持断点续跑
# =====================================================
num_batches = math.ceil(len(post_df) / BATCH_SIZE)
print("总批次数:", num_batches)

for batch_idx in range(num_batches):
    result_file = BATCH_RESULTS_DIR / f"batch_{batch_idx + 1:05d}_result.csv"
    raw_file = RAW_DIR / f"batch_{batch_idx + 1:05d}_raw.txt"

    if result_file.exists():
        print(f"跳过 batch {batch_idx + 1}/{num_batches}（已存在）")
        continue

    start = batch_idx * BATCH_SIZE
    end = min((batch_idx + 1) * BATCH_SIZE, len(post_df))
    batch_df = post_df.iloc[start:end].copy()

    records = build_post_records(batch_df)
    user_prompt = make_user_prompt(records)

    print(f"\n开始 batch {batch_idx + 1}/{num_batches}, posts={len(records)}")

    success = False
    for attempt in range(MAX_RETRY):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_completion_tokens=3000
            )

            raw_text = response.choices[0].message.content

            with open(raw_file, "w", encoding="utf-8") as f:
                f.write(raw_text)

            rows = parse_results(raw_text)
            result_df = pd.DataFrame(rows)
            result_df.to_csv(result_file, index=False, encoding="utf-8-sig")

            print(f"batch {batch_idx + 1} 完成，返回 {len(result_df)} 条")
            success = True
            break

        except Exception as e:
            print(f"batch {batch_idx + 1} 第 {attempt + 1} 次失败：{e}")
            time.sleep(SLEEP_SECONDS)

    if not success:
        raise RuntimeError(f"batch {batch_idx + 1} 连续失败，请检查。")

# =====================================================
# 8. 合并 batch 结果
# =====================================================
# =====================================================
# 合并所有 post batch 结果：跳过空文件和坏文件
# =====================================================
batch_files = sorted(BATCH_RESULTS_DIR.glob("batch_*_result.csv"))

valid_dfs = []
empty_files = []

for f in batch_files:
    try:
        if f.stat().st_size == 0:
            empty_files.append(f)
            continue

        temp = pd.read_csv(f, low_memory=False)

        if temp.empty:
            empty_files.append(f)
            continue

        valid_dfs.append(temp)

    except pd.errors.EmptyDataError:
        empty_files.append(f)
        continue

print("batch 文件总数：", len(batch_files))
print("有效 batch 文件数：", len(valid_dfs))
print("空/坏 batch 文件数：", len(empty_files))

if empty_files:
    print("\n以下 batch 文件是空的或坏的，需要后面补跑：")
    for f in empty_files[:20]:
        print(f)
    if len(empty_files) > 20:
        print("还有更多空文件未显示...")

if not valid_dfs:
    raise ValueError("没有任何有效 batch 结果文件，不能合并。")

batch_result_df = pd.concat(valid_dfs, ignore_index=True)

batch_result_df["post_id"] = batch_result_df["post_id"].astype(str).str.strip()
batch_result_df = batch_result_df.drop_duplicates(subset=["post_id"])

print("\n当前已拿到标签的 post 数：", len(batch_result_df))

# =====================================================
# 9. 找漏分 post，自动补跑
# =====================================================
labeled_ids = set(batch_result_df["post_id"].tolist())
missing_df = post_df[~post_df["post_id"].isin(labeled_ids)].copy().reset_index(drop=True)

print("当前漏分 post 数:", len(missing_df))
missing_df.to_csv(MISSING_FILE, index=False, encoding="utf-8-sig")

if len(missing_df) > 0:
    recovery_batches = math.ceil(len(missing_df) / RECOVERY_BATCH_SIZE)

    print("\n开始补跑漏分 post...")

    for batch_idx in range(recovery_batches):
        recovery_file = RECOVERY_DIR / f"recovery_{batch_idx + 1:05d}.csv"
        recovery_raw_file = RECOVERY_DIR / f"recovery_{batch_idx + 1:05d}_raw.txt"

        if recovery_file.exists():
            print(f"跳过 recovery {batch_idx + 1}/{recovery_batches}（已存在）")
            continue

        start = batch_idx * RECOVERY_BATCH_SIZE
        end = min((batch_idx + 1) * RECOVERY_BATCH_SIZE, len(missing_df))
        sub_df = missing_df.iloc[start:end].copy()

        records = build_post_records(sub_df)
        user_prompt = make_user_prompt(records)

        success = False
        for attempt in range(MAX_RETRY):
            try:
                response = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_completion_tokens=2000
                )

                raw_text = response.choices[0].message.content

                with open(recovery_raw_file, "w", encoding="utf-8") as f:
                    f.write(raw_text)

                rows = parse_results(raw_text)
                recovery_df = pd.DataFrame(rows)
                recovery_df.to_csv(recovery_file, index=False, encoding="utf-8-sig")

                print(f"recovery {batch_idx + 1}/{recovery_batches} 完成，返回 {len(recovery_df)} 条")
                success = True
                break

            except Exception as e:
                print(f"recovery {batch_idx + 1} 第 {attempt + 1} 次失败：{e}")
                time.sleep(SLEEP_SECONDS)

        if not success:
            print(f"recovery {batch_idx + 1} 最终失败，保留为空。")

# =====================================================
# 10. 汇总 recovery 结果
# =====================================================
# =====================================================
# 读取 recovery 结果：跳过空文件和坏文件
# =====================================================
recovery_files = sorted(RECOVERY_DIR.glob("recovery_*.csv"))

recovery_valid_dfs = []
bad_recovery_files = []

for f in recovery_files:
    try:
        if f.stat().st_size == 0:
            bad_recovery_files.append(f)
            continue

        temp = pd.read_csv(f, low_memory=False)

        if temp.empty or len(temp.columns) == 0:
            bad_recovery_files.append(f)
            continue

        recovery_valid_dfs.append(temp)

    except pd.errors.EmptyDataError:
        bad_recovery_files.append(f)
        continue

print("recovery 文件总数：", len(recovery_files))
print("有效 recovery 文件数：", len(recovery_valid_dfs))
print("空/坏 recovery 文件数：", len(bad_recovery_files))

if bad_recovery_files:
    print("空/坏 recovery 文件示例：")
    for f in bad_recovery_files[:10]:
        print(f)

if recovery_valid_dfs:
    recovery_result_df = pd.concat(recovery_valid_dfs, ignore_index=True)
    recovery_result_df["post_id"] = recovery_result_df["post_id"].astype(str).str.strip()
    recovery_result_df = recovery_result_df.drop_duplicates(subset=["post_id"])
else:
    recovery_result_df = pd.DataFrame()

# =====================================================
# 合并 batch 结果 + recovery 结果
# post 版本
# =====================================================
if not recovery_result_df.empty:
    all_results_df = pd.concat([batch_result_df, recovery_result_df], ignore_index=True)
    all_results_df["post_id"] = all_results_df["post_id"].astype(str).str.strip()
    all_results_df = all_results_df.drop_duplicates(subset=["post_id"])
else:
    all_results_df = batch_result_df.copy()

print("\n最终已分类 post 数:", len(all_results_df))
print("最终漏分 post 数:", len(post_df) - len(all_results_df))

print("\n最终已分类 post 数:", len(all_results_df))
print("最终漏分 post 数:", len(post_df) - len(all_results_df))

# =====================================================
# 11. 合并回 post_base
# =====================================================
final_df = post_df.merge(
    all_results_df[
        ["post_id", "post_cluster_id", "post_cluster_label", "post_cluster_keywords", "confidence"]
    ],
    on="post_id",
    how="left"
)

final_df.to_csv(FINAL_RESULT_FILE, index=False, encoding="utf-8-sig")
print("\n全量 post 分类结果已保存:", FINAL_RESULT_FILE)

# =====================================================
# 12. 数量汇总
# =====================================================
count_df = (
    final_df.groupby(["post_cluster_id", "post_cluster_label"], dropna=False)
    .agg(
        post_count=("post_id", "count"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
    .sort_values("post_count", ascending=False)
)

count_df["ratio"] = count_df["post_count"] / len(final_df)
count_df.to_csv(COUNT_FILE, index=False, encoding="utf-8-sig")
print("post cluster 数量汇总已保存:", COUNT_FILE)

# =====================================================
# 13. 情感汇总
# =====================================================
for col in ["avg_sentiment_score", "pos_ratio", "neg_ratio", "neu_ratio", "blame_ratio"]:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

sentiment_summary = (
    final_df.groupby(["post_cluster_id", "post_cluster_label"], dropna=False)
    .agg(
        post_count=("post_id", "count"),
        avg_sentiment_score=("avg_sentiment_score", "mean"),
        avg_pos_ratio=("pos_ratio", "mean"),
        avg_neg_ratio=("neg_ratio", "mean"),
        avg_neu_ratio=("neu_ratio", "mean"),
        main_fine_grained_emotion=("main_fine_grained_emotion", mode_nonempty)
    )
    .reset_index()
    .sort_values("post_cluster_id")
)

sentiment_summary.to_csv(SENTIMENT_FILE, index=False, encoding="utf-8-sig")
print("post cluster 情感汇总已保存:", SENTIMENT_FILE)

# =====================================================
# 14. blame 汇总
# =====================================================
blame_summary = (
    final_df.groupby(["post_cluster_id", "post_cluster_label"], dropna=False)
    .agg(
        post_count=("post_id", "count"),
        avg_blame_ratio=("blame_ratio", "mean"),
        main_blame_subject=("main_blame_subject", mode_nonempty),
        main_blame_object=("main_blame_object", mode_nonempty),
        main_blame_intensity=("main_blame_intensity", mode_nonempty)
    )
    .reset_index()
    .sort_values("post_cluster_id")
)

blame_summary.to_csv(BLAME_FILE, index=False, encoding="utf-8-sig")
print("post cluster blame 汇总已保存:", BLAME_FILE)

# =====================================================
# 15. 打印结果
# =====================================================
print("\n=== post 全量 reaction clustering 完成 ===")
print("总 post 数:", len(final_df))
print("已分类 post 数:", final_df["post_cluster_id"].notna().sum())
print("未分类 post 数:", final_df["post_cluster_id"].isna().sum())

print("\n输出文件：")
print("1.", FINAL_RESULT_FILE)
print("2.", COUNT_FILE)
print("3.", SENTIMENT_FILE)
print("4.", BLAME_FILE)
print("5.", MISSING_FILE)

post_base: (8307, 17)
framework: (12, 7)
清洗后 post 数: 8307
总批次数: 1039
跳过 batch 1/1039（已存在）
跳过 batch 2/1039（已存在）
跳过 batch 3/1039（已存在）
跳过 batch 4/1039（已存在）
跳过 batch 5/1039（已存在）
跳过 batch 6/1039（已存在）
跳过 batch 7/1039（已存在）
跳过 batch 8/1039（已存在）
跳过 batch 9/1039（已存在）
跳过 batch 10/1039（已存在）
跳过 batch 11/1039（已存在）
跳过 batch 12/1039（已存在）
跳过 batch 13/1039（已存在）
跳过 batch 14/1039（已存在）
跳过 batch 15/1039（已存在）
跳过 batch 16/1039（已存在）
跳过 batch 17/1039（已存在）
跳过 batch 18/1039（已存在）
跳过 batch 19/1039（已存在）
跳过 batch 20/1039（已存在）
跳过 batch 21/1039（已存在）
跳过 batch 22/1039（已存在）
跳过 batch 23/1039（已存在）
跳过 batch 24/1039（已存在）
跳过 batch 25/1039（已存在）
跳过 batch 26/1039（已存在）
跳过 batch 27/1039（已存在）
跳过 batch 28/1039（已存在）
跳过 batch 29/1039（已存在）
跳过 batch 30/1039（已存在）
跳过 batch 31/1039（已存在）
跳过 batch 32/1039（已存在）
跳过 batch 33/1039（已存在）
跳过 batch 34/1039（已存在）
跳过 batch 35/1039（已存在）
跳过 batch 36/1039（已存在）
跳过 batch 37/1039（已存在）
跳过 batch 38/1039（已存在）
跳过 batch 39/1039（已存在）
跳过 batch 40/1039（已存在）
跳过 batch 41/1039（已存在）
跳过 batch 42/1039（已存在）
跳过 batch 43/1039

In [20]:
import pandas as pd
from pathlib import Path

# =====================================================
# 1. 路径设置
# =====================================================
BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")

POST_FILE = BASE_DIR / "结果" / "最后跑出来的结果---post" / "full_post_reaction_results.csv"
COMMENT_FILE = BASE_DIR / "结果" / "最后跑出来的结果---comment" / "full_comment_cluster_results.csv"

OUTPUT_DIR = BASE_DIR / "结果"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MERGED_FILE = OUTPUT_DIR / "topic_merged_final.csv"
DIST_FILE = OUTPUT_DIR / "post_cluster_comment_cluster_distribution.csv"
SENTIMENT_FILE = OUTPUT_DIR / "post_cluster_sentiment_summary.csv"
BLAME_FILE = OUTPUT_DIR / "post_cluster_blame_summary.csv"
PIVOT_FILE = OUTPUT_DIR / "post_comment_cluster_pivot.csv"

# =====================================================
# 2. 读取数据
# =====================================================
post_df = pd.read_csv(POST_FILE, low_memory=False)
comment_df = pd.read_csv(COMMENT_FILE, low_memory=False)

print("post 表大小：", post_df.shape)
print("comment 表大小：", comment_df.shape)

# =====================================================
# 3. 标准化 post 表字段
# =====================================================
post_df["post_id"] = post_df["post_id"].astype(str).str.strip()

# 如果 post 表里叫 cluster_id / cluster_label，就改名
if "cluster_id" in post_df.columns:
    post_df = post_df.rename(columns={"cluster_id": "post_cluster_id"})
if "cluster_label" in post_df.columns:
    post_df = post_df.rename(columns={"cluster_label": "post_cluster_label"})

# 如果本来已经叫 post_cluster_id / post_cluster_label，就不用改
post_keep = ["post_id", "post_cluster_id", "post_cluster_label"]
post_topic = post_df[post_keep].drop_duplicates(subset=["post_id"]).copy()

# 处理未分类 post
post_topic["post_cluster_id"] = post_topic["post_cluster_id"].fillna(99)
post_topic["post_cluster_label"] = post_topic["post_cluster_label"].fillna("其他/未分类")

# =====================================================
# 4. 标准化 comment 表字段
# =====================================================
comment_df["post_id"] = comment_df["post_id"].astype(str).str.strip()
comment_df["comment_id"] = comment_df["comment_id"].astype(str).str.strip()

# 如果 comment 表里叫 cluster_id / cluster_label，就改名
if "cluster_id" in comment_df.columns:
    comment_df = comment_df.rename(columns={"cluster_id": "comment_cluster_id"})
if "cluster_label" in comment_df.columns:
    comment_df = comment_df.rename(columns={"cluster_label": "comment_cluster_label"})

# 处理未分类 comment
comment_df["comment_cluster_id"] = comment_df["comment_cluster_id"].fillna(99)
comment_df["comment_cluster_label"] = comment_df["comment_cluster_label"].fillna("其他/未分类")

# =====================================================
# 5. 合并 post cluster + comment cluster
# =====================================================
topic_merged = comment_df.merge(
    post_topic,
    on="post_id",
    how="left"
)

# 如果有些 comment 的 post 没在 post 表里
topic_merged["post_cluster_id"] = topic_merged["post_cluster_id"].fillna(99)
topic_merged["post_cluster_label"] = topic_merged["post_cluster_label"].fillna("其他/未分类")

# 调整核心列顺序
front_cols = [
    "comment_id",
    "post_id",
    "post_cluster_id",
    "post_cluster_label",
    "comment_cluster_id",
    "comment_cluster_label"
]

other_cols = [c for c in topic_merged.columns if c not in front_cols]
topic_merged = topic_merged[front_cols + other_cols]

topic_merged.to_csv(MERGED_FILE, index=False, encoding="utf-8-sig")
print("合并总表已保存：", MERGED_FILE)

# =====================================================
# 6. 表 1：post cluster × comment cluster 分布
# =====================================================
dist = (
    topic_merged
    .groupby(
        ["post_cluster_id", "post_cluster_label", "comment_cluster_id", "comment_cluster_label"],
        dropna=False
    )
    .agg(comment_count=("comment_id", "count"))
    .reset_index()
)

# 计算在每个 post cluster 内部的占比
dist["total_comments_in_post_cluster"] = dist.groupby(
    ["post_cluster_id", "post_cluster_label"]
)["comment_count"].transform("sum")

dist["ratio_within_post_cluster"] = (
    dist["comment_count"] / dist["total_comments_in_post_cluster"]
)

dist = dist.sort_values(
    ["post_cluster_id", "comment_count"],
    ascending=[True, False]
)

dist.to_csv(DIST_FILE, index=False, encoding="utf-8-sig")
print("post cluster × comment cluster 分布表已保存：", DIST_FILE)

# =====================================================
# 7. 表 2：post cluster 情感汇总
# =====================================================
topic_merged["sentiment_score"] = pd.to_numeric(
    topic_merged["sentiment_score"], errors="coerce"
)

sentiment_pivot = (
    topic_merged
    .pivot_table(
        index=["post_cluster_id", "post_cluster_label"],
        columns="sentiment_category",
        values="comment_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for col in ["negative", "neutral", "positive"]:
    if col not in sentiment_pivot.columns:
        sentiment_pivot[col] = 0

sentiment_pivot["total_comments"] = (
    sentiment_pivot["negative"] +
    sentiment_pivot["neutral"] +
    sentiment_pivot["positive"]
).replace(0, pd.NA)

sentiment_pivot["negative_ratio"] = sentiment_pivot["negative"] / sentiment_pivot["total_comments"]
sentiment_pivot["neutral_ratio"] = sentiment_pivot["neutral"] / sentiment_pivot["total_comments"]
sentiment_pivot["positive_ratio"] = sentiment_pivot["positive"] / sentiment_pivot["total_comments"]

avg_sentiment = (
    topic_merged
    .groupby(["post_cluster_id", "post_cluster_label"], as_index=False)
    .agg(
        comment_count=("comment_id", "count"),
        avg_sentiment_score=("sentiment_score", "mean")
    )
)

sentiment_summary = avg_sentiment.merge(
    sentiment_pivot[
        [
            "post_cluster_id",
            "post_cluster_label",
            "negative",
            "neutral",
            "positive",
            "negative_ratio",
            "neutral_ratio",
            "positive_ratio"
        ]
    ],
    on=["post_cluster_id", "post_cluster_label"],
    how="left"
)

sentiment_summary = sentiment_summary.sort_values("comment_count", ascending=False)
sentiment_summary.to_csv(SENTIMENT_FILE, index=False, encoding="utf-8-sig")
print("post cluster 情感汇总表已保存：", SENTIMENT_FILE)

# =====================================================
# 8. 表 3：post cluster blame 汇总
# =====================================================
def mode_nonempty(series):
    s = series.dropna().astype(str).str.strip()
    s = s[~s.isin(["", "nan", "None", "null"])]
    if len(s) == 0:
        return ""
    return s.value_counts().index[0]

def blame_ratio_func(series):
    s = series.fillna("").astype(str).str.lower().str.strip()
    return (s == "yes").mean()

blame_summary = (
    topic_merged
    .groupby(["post_cluster_id", "post_cluster_label"], as_index=False)
    .agg(
        comment_count=("comment_id", "count"),
        blame_ratio=("has_blame", blame_ratio_func),
        main_blame_subject=("blame_subject", mode_nonempty),
        main_blame_object=("blame_object", mode_nonempty),
        main_blame_intensity=("blame_intensity", mode_nonempty)
    )
)

blame_summary = blame_summary.sort_values("comment_count", ascending=False)
blame_summary.to_csv(BLAME_FILE, index=False, encoding="utf-8-sig")
print("post cluster blame 汇总表已保存：", BLAME_FILE)

# =====================================================
# 9. 宽表：post cluster × comment cluster pivot
#    适合画热力图 / 堆叠图
# =====================================================
pivot = (
    dist
    .pivot_table(
        index=["post_cluster_id", "post_cluster_label"],
        columns="comment_cluster_label",
        values="ratio_within_post_cluster",
        fill_value=0
    )
    .reset_index()
)

pivot.to_csv(PIVOT_FILE, index=False, encoding="utf-8-sig")
print("post-comment cluster pivot 表已保存：", PIVOT_FILE)

# =====================================================
# 10. 简单检查
# =====================================================
print("\n=== 完成 ===")
print("合并后总行数：", len(topic_merged))
print("post cluster 数：", topic_merged["post_cluster_label"].nunique())
print("comment cluster 数：", topic_merged["comment_cluster_label"].nunique())

print("\n输出文件：")
print("1.", MERGED_FILE)
print("2.", DIST_FILE)
print("3.", SENTIMENT_FILE)
print("4.", BLAME_FILE)
print("5.", PIVOT_FILE)

post 表大小： (8307, 21)
comment 表大小： (69879, 13)
合并总表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/topic_merged_final.csv
post cluster × comment cluster 分布表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_cluster_comment_cluster_distribution.csv
post cluster 情感汇总表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_cluster_sentiment_summary.csv
post cluster blame 汇总表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_cluster_blame_summary.csv
post-comment cluster pivot 表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_comment_cluster_pivot.csv

=== 完成 ===
合并后总行数： 69879
post cluster 数： 13
comment cluster 数： 13

输出文件：
1. /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/topic_merged_final.csv
2. /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_cluster_comment_cluster_distribution.csv
3. /Users/xujingyu/Desktop/课程资料/5508/pr

In [21]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果")

merged = pd.read_csv(BASE_DIR / "topic_merged_final.csv", low_memory=False)

other = merged[merged["post_cluster_label"] == "其他/未分类"]

print("其他/未分类评论数：", len(other))
print("涉及post数：", other["post_id"].nunique())

print(other["post_id"].value_counts().head(20))

其他/未分类评论数： 5426
涉及post数： 2498
post_id
LK_20251228040007296_a2f86a18          22
LK_20251227040004784_68564f3b          22
LK_20251228040007196_5520590f          21
LK_20251228040007139_d67f8a0a          21
LK_20251227040004841_c44710ed          21
LK_20251226040005298_ab46290b          19
LK_20251228040007085_4cb980e1          17
LK_20251224040004381_ed809f26          12
LK_20251227040004689_64571ff5          12
LK_20251226040005248_c62c303b          11
Threads_1773766831_ad9a4385             9
LK_20251227040004885_6e23c740           4
FACEBOOK_20260218031120535_1f34b9c5     3
FACEBOOK_20260107040056644_79a4a81c     3
FACEBOOK_20260205031548382_9a5144b5     3
FACEBOOK_20260218031120800_c5d3b9e0     3
FACEBOOK_20260218031120909_577b6a7b     3
FACEBOOK_20260107040050805_9a796c3f     3
FACEBOOK_20260218031117114_c29978f8     3
FACEBOOK_20260107040052369_30243fc8     3
Name: count, dtype: int64


In [22]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")

POST_RESULT_FILE = BASE_DIR / "结果" / "最后跑出来的结果---post" / "full_post_reaction_results.csv"
COMMENT_RESULT_FILE = BASE_DIR / "结果" / "最后跑出来的结果---comment" / "full_comment_cluster_results.csv"
POST_BASE_FILE = BASE_DIR / "post_base.csv"

OUTPUT_DIR = BASE_DIR / "结果"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MISSING_POST_FILE = OUTPUT_DIR / "missing_posts_need_post_cluster.csv"

post_result = pd.read_csv(POST_RESULT_FILE, low_memory=False)
comment_result = pd.read_csv(COMMENT_RESULT_FILE, low_memory=False)
post_base = pd.read_csv(POST_BASE_FILE, low_memory=False)

post_result["post_id"] = post_result["post_id"].astype(str).str.strip()
comment_result["post_id"] = comment_result["post_id"].astype(str).str.strip()
post_base["post_id"] = post_base["post_id"].astype(str).str.strip()

# comment 里出现过的 post
comment_post_ids = set(comment_result["post_id"].dropna())

# post 聚类结果里已经有的 post
classified_post_ids = set(post_result["post_id"].dropna())

# 缺失 post
missing_ids = comment_post_ids - classified_post_ids

print("comment里出现的post数：", len(comment_post_ids))
print("post聚类结果里的post数：", len(classified_post_ids))
print("需要补跑的post数：", len(missing_ids))

missing_posts = post_base[post_base["post_id"].isin(missing_ids)].copy()

print("能在post_base里找到文本的缺失post数：", len(missing_posts))

missing_posts.to_csv(MISSING_POST_FILE, index=False, encoding="utf-8-sig")

print("已保存：", MISSING_POST_FILE)

comment里出现的post数： 10606
post聚类结果里的post数： 8307
需要补跑的post数： 2459
能在post_base里找到文本的缺失post数： 0
已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/missing_posts_need_post_cluster.csv


In [23]:
import pandas as pd
from pathlib import Path

# =====================================================
# 1. 路径设置
# =====================================================
BASE_DIR = Path("/Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs")

POST_FILE = BASE_DIR / "结果" / "最后跑出来的结果---post" / "full_post_reaction_results.csv"
COMMENT_FILE = BASE_DIR / "结果" / "最后跑出来的结果---comment" / "full_comment_cluster_results.csv"

OUTPUT_DIR = BASE_DIR / "结果"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_MERGED_FILE = OUTPUT_DIR / "topic_merged_final_all_comments.csv"
CORE_MERGED_FILE = OUTPUT_DIR / "topic_merged_final_core_only.csv"

DIST_ALL_FILE = OUTPUT_DIR / "post_comment_cluster_distribution_all.csv"
DIST_CORE_FILE = OUTPUT_DIR / "post_comment_cluster_distribution_core.csv"

SENTIMENT_CORE_FILE = OUTPUT_DIR / "post_cluster_sentiment_summary_core.csv"
BLAME_CORE_FILE = OUTPUT_DIR / "post_cluster_blame_summary_core.csv"
PIVOT_CORE_FILE = OUTPUT_DIR / "post_comment_cluster_pivot_core.csv"

UNMATCHED_POST_FILE = OUTPUT_DIR / "unmatched_post_ids_from_comment.csv"

# =====================================================
# 2. 读取数据
# =====================================================
post_df = pd.read_csv(POST_FILE, low_memory=False)
comment_df = pd.read_csv(COMMENT_FILE, low_memory=False)

print("post 表大小：", post_df.shape)
print("comment 表大小：", comment_df.shape)

# =====================================================
# 3. 标准化字段
# =====================================================
post_df["post_id"] = post_df["post_id"].astype(str).str.strip()
comment_df["post_id"] = comment_df["post_id"].astype(str).str.strip()
comment_df["comment_id"] = comment_df["comment_id"].astype(str).str.strip()

# post 表字段统一
if "cluster_id" in post_df.columns:
    post_df = post_df.rename(columns={"cluster_id": "post_cluster_id"})
if "cluster_label" in post_df.columns:
    post_df = post_df.rename(columns={"cluster_label": "post_cluster_label"})

# comment 表字段统一
if "cluster_id" in comment_df.columns:
    comment_df = comment_df.rename(columns={"cluster_id": "comment_cluster_id"})
if "cluster_label" in comment_df.columns:
    comment_df = comment_df.rename(columns={"cluster_label": "comment_cluster_label"})

# 去重
post_topic = post_df[
    ["post_id", "post_cluster_id", "post_cluster_label"]
].drop_duplicates(subset=["post_id"]).copy()

comment_df = comment_df.drop_duplicates(subset=["comment_id"]).copy()

# =====================================================
# 4. 检查匹配情况
# =====================================================
comment_post_ids = set(comment_df["post_id"].dropna())
post_result_ids = set(post_topic["post_id"].dropna())

missing_post_ids = comment_post_ids - post_result_ids

print("\ncomment 里出现的 post 数：", len(comment_post_ids))
print("post 聚类结果里的 post 数：", len(post_result_ids))
print("comment 中找不到 post cluster 的 post 数：", len(missing_post_ids))

unmatched_posts = comment_df[comment_df["post_id"].isin(missing_post_ids)][["post_id"]].drop_duplicates()
unmatched_posts.to_csv(UNMATCHED_POST_FILE, index=False, encoding="utf-8-sig")
print("未匹配 post_id 已保存：", UNMATCHED_POST_FILE)

# =====================================================
# 5. 合并 post cluster + comment cluster
# =====================================================
merged = comment_df.merge(
    post_topic,
    on="post_id",
    how="left"
)

# 标记是否成功匹配 post cluster
merged["post_cluster_matched"] = merged["post_cluster_id"].notna()

# comment 未分类处理
merged["comment_cluster_id"] = merged["comment_cluster_id"].fillna(99)
merged["comment_cluster_label"] = merged["comment_cluster_label"].fillna("其他/未分类")

# post 未匹配处理
merged["post_cluster_id"] = merged["post_cluster_id"].fillna(99)
merged["post_cluster_label"] = merged["post_cluster_label"].fillna("缺少post文本/未参与post聚类")

# 核心列提前
front_cols = [
    "comment_id",
    "post_id",
    "post_cluster_id",
    "post_cluster_label",
    "post_cluster_matched",
    "comment_cluster_id",
    "comment_cluster_label"
]

other_cols = [c for c in merged.columns if c not in front_cols]
merged = merged[front_cols + other_cols]

# 保存完整总表
merged.to_csv(FULL_MERGED_FILE, index=False, encoding="utf-8-sig")
print("\n完整合并总表已保存：", FULL_MERGED_FILE)

# =====================================================
# 6. 生成核心分析表：只保留成功匹配 post cluster 的评论
# =====================================================
core = merged[merged["post_cluster_matched"] == True].copy()

# 如果 post 本身是 99，也排除
core = core[core["post_cluster_id"] != 99].copy()

core.to_csv(CORE_MERGED_FILE, index=False, encoding="utf-8-sig")
print("核心分析表已保存：", CORE_MERGED_FILE)

print("\n完整表 comment 数：", len(merged))
print("核心表 comment 数：", len(core))
print("被排除 comment 数：", len(merged) - len(core))

# =====================================================
# 7. post cluster × comment cluster 分布函数
# =====================================================
def make_distribution(df, output_file):
    dist = (
        df.groupby(
            [
                "post_cluster_id",
                "post_cluster_label",
                "comment_cluster_id",
                "comment_cluster_label"
            ],
            dropna=False
        )
        .agg(comment_count=("comment_id", "count"))
        .reset_index()
    )

    dist["total_comments_in_post_cluster"] = dist.groupby(
        ["post_cluster_id", "post_cluster_label"]
    )["comment_count"].transform("sum")

    dist["ratio_within_post_cluster"] = (
        dist["comment_count"] / dist["total_comments_in_post_cluster"]
    )

    dist = dist.sort_values(
        ["post_cluster_id", "comment_count"],
        ascending=[True, False]
    )

    dist.to_csv(output_file, index=False, encoding="utf-8-sig")
    return dist

dist_all = make_distribution(merged, DIST_ALL_FILE)
dist_core = make_distribution(core, DIST_CORE_FILE)

print("\n完整分布表已保存：", DIST_ALL_FILE)
print("核心分布表已保存：", DIST_CORE_FILE)

# =====================================================
# 8. 核心表：post cluster 情感汇总
# =====================================================
core["sentiment_score"] = pd.to_numeric(core["sentiment_score"], errors="coerce")

sentiment_pivot = (
    core.pivot_table(
        index=["post_cluster_id", "post_cluster_label"],
        columns="sentiment_category",
        values="comment_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for col in ["negative", "neutral", "positive"]:
    if col not in sentiment_pivot.columns:
        sentiment_pivot[col] = 0

sentiment_pivot["total_comments"] = (
    sentiment_pivot["negative"] +
    sentiment_pivot["neutral"] +
    sentiment_pivot["positive"]
).replace(0, pd.NA)

sentiment_pivot["negative_ratio"] = sentiment_pivot["negative"] / sentiment_pivot["total_comments"]
sentiment_pivot["neutral_ratio"] = sentiment_pivot["neutral"] / sentiment_pivot["total_comments"]
sentiment_pivot["positive_ratio"] = sentiment_pivot["positive"] / sentiment_pivot["total_comments"]

avg_sentiment = (
    core.groupby(["post_cluster_id", "post_cluster_label"], as_index=False)
    .agg(
        comment_count=("comment_id", "count"),
        avg_sentiment_score=("sentiment_score", "mean")
    )
)

sentiment_summary = avg_sentiment.merge(
    sentiment_pivot[
        [
            "post_cluster_id",
            "post_cluster_label",
            "negative",
            "neutral",
            "positive",
            "negative_ratio",
            "neutral_ratio",
            "positive_ratio"
        ]
    ],
    on=["post_cluster_id", "post_cluster_label"],
    how="left"
)

sentiment_summary = sentiment_summary.sort_values("comment_count", ascending=False)
sentiment_summary.to_csv(SENTIMENT_CORE_FILE, index=False, encoding="utf-8-sig")

print("核心 post cluster 情感汇总已保存：", SENTIMENT_CORE_FILE)

# =====================================================
# 9. 核心表：post cluster blame 汇总
# =====================================================
def mode_nonempty(series):
    s = series.dropna().astype(str).str.strip()
    s = s[~s.isin(["", "nan", "None", "null"])]
    if len(s) == 0:
        return ""
    return s.value_counts().index[0]

def blame_ratio_func(series):
    s = series.fillna("").astype(str).str.lower().str.strip()
    return (s == "yes").mean()

blame_summary = (
    core.groupby(["post_cluster_id", "post_cluster_label"], as_index=False)
    .agg(
        comment_count=("comment_id", "count"),
        blame_ratio=("has_blame", blame_ratio_func),
        main_blame_subject=("blame_subject", mode_nonempty),
        main_blame_object=("blame_object", mode_nonempty),
        main_blame_intensity=("blame_intensity", mode_nonempty)
    )
)

blame_summary = blame_summary.sort_values("comment_count", ascending=False)
blame_summary.to_csv(BLAME_CORE_FILE, index=False, encoding="utf-8-sig")

print("核心 post cluster blame 汇总已保存：", BLAME_CORE_FILE)

# =====================================================
# 10. 核心 pivot 表：适合热力图
# =====================================================
pivot_core = (
    dist_core.pivot_table(
        index=["post_cluster_id", "post_cluster_label"],
        columns="comment_cluster_label",
        values="ratio_within_post_cluster",
        fill_value=0
    )
    .reset_index()
)

pivot_core.to_csv(PIVOT_CORE_FILE, index=False, encoding="utf-8-sig")
print("核心 pivot 表已保存：", PIVOT_CORE_FILE)

# =====================================================
# 11. 最终检查
# =====================================================
print("\n=== 重新合并完成 ===")
print("完整表 comment 数：", len(merged))
print("核心表 comment 数：", len(core))
print("完整表 post cluster 数：", merged["post_cluster_label"].nunique())
print("核心表 post cluster 数：", core["post_cluster_label"].nunique())
print("comment cluster 数：", merged["comment_cluster_label"].nunique())

print("\n输出文件：")
print("1.", FULL_MERGED_FILE)
print("2.", CORE_MERGED_FILE)
print("3.", DIST_ALL_FILE)
print("4.", DIST_CORE_FILE)
print("5.", SENTIMENT_CORE_FILE)
print("6.", BLAME_CORE_FILE)
print("7.", PIVOT_CORE_FILE)
print("8.", UNMATCHED_POST_FILE)

post 表大小： (8307, 21)
comment 表大小： (69879, 13)

comment 里出现的 post 数： 10606
post 聚类结果里的 post 数： 8307
comment 中找不到 post cluster 的 post 数： 2459
未匹配 post_id 已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/unmatched_post_ids_from_comment.csv

完整合并总表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/topic_merged_final_all_comments.csv
核心分析表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/topic_merged_final_core_only.csv

完整表 comment 数： 69879
核心表 comment 数： 64453
被排除 comment 数： 5426

完整分布表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_comment_cluster_distribution_all.csv
核心分布表已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_comment_cluster_distribution_core.csv
核心 post cluster 情感汇总已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/part2_outputs/结果/post_cluster_sentiment_summary_core.csv
核心 post cluster blame 汇总已保存： /Users/xujingyu/Desktop/课程资料/5508/project2/表2-要用的/p